<a href="https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatimaali123-ai/flyrank-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Distributions

I first inspect the distributions of key search and content signals before interpreting their relationship with content decline. Search and traffic variables are expected to be unevenly distributed, so summary statistics and quantiles are used instead of relying only on averages.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the starter dataset
url = "https://raw.githubusercontent.com/fatimaali123-ai/flyrank-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

key_signals = [
    "impressions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "ctr",
    "avg_position",
    "content_age_days"
]

available_signals = [c for c in key_signals if c in df.columns]

print("Dataset shape:", df.shape)
print("\nKey signal summary:")
display(df[available_signals].describe().T)

for col in available_signals:
    print(f"\n{col}")
    print(df[col].quantile([0, .25, .5, .75, .90, .95, 1]))


Dataset shape: (30000, 44)

Key signal summary:


,count,mean,std,min,25%,50%,75%,max
impressions_90d,30000.0,5200.366300,16838.019547,1.0,81.0,731.00,3615.25,517715.0
impressions_last_30d,30000.0,1429.058733,5643.852081,0.0,10.0,139.00,768.00,238796.0
clicks_last_30d,30000.0,4.933867,23.929393,0.0,0.0,0.00,2.00,1176.0
ctr,30000.0,0.510733,3.279162,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342380,15.216790,0.0,6.2,10.80,22.30,245.0
content_age_days,30000.0,256.167800,132.707930,90.0,132.0,236.00,333.00,564.0



impressions_90d
0.00         1.00
0.25        81.00
0.50       731.00
0.75      3615.25
0.90     12136.40
0.95     22996.50
1.00    517715.00
Name: impressions_90d, dtype: float64

impressions_last_30d
0.00         0.0
0.25        10.0
0.50       139.0
0.75       768.0
0.90      2991.1
0.95      6313.1
1.00    238796.0
Name: impressions_last_30d, dtype: float64

clicks_last_30d
0.00       0.0
0.25       0.0
0.50       0.0
0.75       2.0
0.90      10.0
0.95      21.0
1.00    1176.0
Name: clicks_last_30d, dtype: float64

ctr
0.00      0.00
0.25      0.00
0.50      0.07
0.75      0.29
0.90      0.65
0.95      1.09
1.00    100.00
Name: ctr, dtype: float64

avg_position
0.00      0.0
0.25      6.2
0.50     10.8
0.75     22.3
0.90     36.8
0.95     48.2
1.00    245.0
Name: avg_position, dtype: float64

content_age_days
0.00     90.0
0.25    132.0
0.50    236.0
0.75    333.0
0.90    463.0
0.95    487.0
1.00    564.0
Name: content_age_days, dtype: float64


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal tests

I test three safe signals using grouped comparisons. The tests compare the observed decline rate across meaningful signal groups.

The signals tested are impressions, CTR, and average position. The verdicts are based on the observed direction of the differences and are treated as directional evidence rather than causal effects.

In [10]:
# Create target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

# Signal 1: impressions
df["impressions_group"] = pd.qcut(
    df["impressions_90d"].rank(method="first"),
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
)

signal1 = df.groupby("impressions_group", observed=True)["is_declining_label"].agg(
    ["mean", "count"]
)

print("SIGNAL #1 — Impressions")
display(signal1)

# Signal 2: CTR
df["ctr_group"] = pd.qcut(
    df["ctr"].rank(method="first"),
    q=4,
    labels=["Low", "Medium-Low", "Medium-High", "High"]
)

signal2 = df.groupby("ctr_group", observed=True)["is_declining_label"].agg(
    ["mean", "count"]
)

print("SIGNAL #2 — CTR")
display(signal2)

# Signal 3: average position
position_df = df[df["avg_position"] > 0].copy()

position_df["position_group"] = pd.qcut(
    position_df["avg_position"].rank(method="first"),
    q=4,
    labels=["Best", "Better", "Worse", "Worst"]
)

signal3 = position_df.groupby(
    "position_group", observed=True
)["is_declining_label"].agg(["mean", "count"])

print("SIGNAL #3 — Average position")
display(signal3)

print("\nDirectional verdicts:")
print("Signal #1 — Impressions: MIXED / directional")
print("Signal #2 — CTR: MIXED / directional")
print("Signal #3 — Average position: MIXED / directional")

SIGNAL #1 — Impressions


,mean,count
impressions_group,,
Low,0.376000,7500
Medium-Low,0.604667,7500
Medium-High,0.625600,7500
High,0.562000,7500


SIGNAL #2 — CTR


,mean,count
ctr_group,,
Low,0.490267,7500
Medium-Low,0.553067,7500
Medium-High,0.605200,7500
High,0.519733,7500


SIGNAL #3 — Average position


,mean,count
position_group,,
Best,0.550632,7199
Better,0.581886,7199
Worse,0.612392,7198
Worst,0.512988,7199



Directional verdicts:
Signal #1 — Impressions: MIXED / directional
Signal #2 — CTR: MIXED / directional
Signal #3 — Average position: MIXED / directional


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag-linked test

The Week-4 baseline uses impressions and CTR as transparent signals for prioritization. I therefore test whether declining content is distributed differently across low and high impression/CTR groups.

The purpose is not to prove that the rule causes decline, but to check whether the flag's underlying assumption is directionally supported by the observed data.

In [11]:
# Flag-linked test: impressions + CTR

df["low_impressions"] = df["impressions_90d"] <= df["impressions_90d"].median()
df["low_ctr"] = df["ctr"] <= df["ctr"].median()

flag_test = (
    df.groupby(["low_impressions", "low_ctr"])["is_declining_label"]
      .agg(["mean", "count"])
      .reset_index()
)

print("Flag-linked test:")
display(flag_test)

# Compare the baseline-style flagged group with everyone else
flagged = df["low_impressions"] & df["low_ctr"]

flagged_rate = df.loc[flagged, "is_declining_label"].mean()
other_rate = df.loc[~flagged, "is_declining_label"].mean()

print(f"Decline rate among low-impression + low-CTR items: {flagged_rate:.3f}")
print(f"Decline rate among other items: {other_rate:.3f}")

if flagged_rate > other_rate:
    verdict = "DIRECTIONALLY SUPPORTED"
elif flagged_rate < other_rate:
    verdict = "OPPOSITE"
else:
    verdict = "MIXED"

print("Flag verdict:", verdict)


Flag-linked test:


,low_impressions,low_ctr,mean,count
0,False,False,0.567007,11275
1,False,True,0.674993,3723
2,True,False,0.540703,3501
3,True,True,0.475002,11501


Decline rate among low-impression + low-CTR items: 0.475
Decline rate among other items: 0.584
Flag verdict: OPPOSITE


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### What this means in practice

The signal audit provides directional evidence about which observable content signals are useful for prioritization, but the signals should not be treated as causal explanations of decline.

The practical use is to combine multiple signals into a review queue and have a human analyst verify the underlying evidence before making a content decision. A signal that is mixed or weak should be treated as supporting context rather than an automatic trigger.

In [12]:
print("PRACTICAL INTERPRETATION")
print("- Use multiple signals rather than relying on one field.")
print("- Treat observed relationships as directional evidence.")
print("- Use scores to prioritize human review.")
print("- Do not interpret the signals as causal drivers.")

PRACTICAL INTERPRETATION
- Use multiple signals rather than relying on one field.
- Treat observed relationships as directional evidence.
- Use scores to prioritize human review.
- Do not interpret the signals as causal drivers.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.